<a href="https://colab.research.google.com/github/nikhitarao/nikhitarao_projects/blob/Healthcare-Insurance-Medical-Inflation-Analysis-using-Synthetic-Data/Inpatient_Healthcare_Medical_Procedure_Synthetic_Data_Generation_Member_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Health Insurance Member Database for a Fictional New York based Health Insurer based on Policy Database

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Import Libraries

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

#### Load the Policy Dataset

In [3]:
policy_df = pd.read_csv("/content/drive/My Drive/My Data Projects/Inpatient_Healthcare_Insurance/inpatient_healthcare_insurance_synthetic_policy_dataset.csv")

#### Initialize the Member Dataset

In [4]:

# Initialize an empty DataFrame for Member Dataset
columns = [
    "Policy ID", "Policyholder ID", "Member ID", "Member Ethnicity", "Member Status",
    "Member Marital Status", "Member Designation", "Member Relationship", "Member Gender",
    "Member Age", "Member Income", "Member Chronic Conditions", "Member Smoking Status",
    "Member Alcohol Consumption", "Member Physical Activity Level", "Member Height",
    "Member Weight", "Member BMI Category", "Member Disabilities", "Member Claim History",
    "Member Claim Frequency", "Member Risk Score", "Member Tenure", "Member Productivity Index",
    "Member Work Hours"
]

member_df = pd.DataFrame(columns=columns)

#### Generate Member Data for Each Policy for columns Policy ID, Policyholder ID, Member ID, Member Ethnicity, Member Status, Member Inclusion Date, Member Removal Date

In [ ]:
# Helper Functions for New Columns
def assign_marital_status(location):
    status_distribution = {
        "Urban": {"Single": 50, "Married": 40, "Divorced": 7, "Widowed": 3},
        "Suburban": {"Single": 45, "Married": 45, "Divorced": 7, "Widowed": 3},
        "Rural": {"Single": 40, "Married": 45, "Divorced": 10, "Widowed": 5}
    }
    choices, weights = zip(*status_distribution[location].items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_designation(policy_type):
    if policy_type == "Group Health Plan":
        choices = ["Executive", "Mid-level Manager", "Entry-level Employee", "Support Staff"]
        weights = [5, 15, 50, 30]
        return np.random.choice(choices, p=np.array(weights) / sum(weights))
    return None

def assign_relationship(num_members, member_index):
    if num_members == 1:
        return "Policyholder"
    elif num_members == 2:
        return "Policyholder" if member_index == 1 else "Spouse"
    else:
        roles = ["Policyholder", "Spouse", "Child"]
        return roles[min(member_index - 1, len(roles) - 1)]

def assign_gender(location, is_child):
    if is_child:
        return np.random.choice(["Male", "Female"], p=[0.5, 0.5])
    gender_distribution = {"Urban": [48.5, 50, 1.5], "Suburban": [49, 50, 1], "Rural": [50, 50, 0]}
    return np.random.choice(["Male", "Female", "Other"], p=np.array(gender_distribution[location]) / 100)

def assign_age(policy_type, relationship):
    if policy_type == "Family Plan":
        if relationship == "Policyholder":
            return np.random.randint(30, 50)
        elif relationship == "Spouse":
            return np.random.randint(25, 50)
        elif relationship == "Child":
            return np.random.randint(0, 18)
        elif relationship == "Parent":
            return np.random.randint(50, 70)
    elif policy_type == "Group Health Plan":
        return np.random.randint(22, 65)
    else:
        return np.random.randint(18, 99)

def assign_income(policy_type, designation, location, age):
    if policy_type == "Group Health Plan":
        income_ranges = {
            "Executive": (100000, 200000),
            "Mid-level Manager": (60000, 100000),
            "Entry-level Employee": (40000, 60000),
            "Support Staff": (20000, 40000)
        }
        return np.random.randint(*income_ranges.get(designation, (0, 0)))
    elif policy_type == "Not a Group":
        income_ranges = {"Urban": (50000, 100000), "Suburban": (40000, 80000), "Rural": (30000, 60000)}
        return np.random.randint(*income_ranges[location]) if age > 18 else None
    return None

def assign_chronic_conditions():
    conditions = ["None", "Asthma", "Diabetes", "Hypertension", "Depression", "Combination"]
    weights = [60, 5, 8, 12, 15, 5]
    return np.random.choice(conditions, p=np.array(weights) / sum(weights))

def assign_smoking_status():
    return np.random.choice(["Smoker", "Non-Smoker", "Ex-Smoker"], p=[0.2, 0.75, 0.05])

def assign_alcohol_consumption():
    return np.random.choice(["None", "Moderate", "High"], p=[0.3, 0.6, 0.1])

# Ensure Policy ID is consistently formatted as a string
policy_df['Policy ID'] = policy_df['Policy ID'].astype(str)

# Generate member_df (fallback sample data)
member_df = policy_df[['Policy ID', 'Policyholder ID']].copy()
member_df['Member ID'] = ['M-' + str(i) for i in range(1, len(member_df) + 1)]
member_df['Member Ethnicity'] = ['Unknown'] * len(member_df)
member_df['Member Status'] = ['Unknown'] * len(member_df)

print("Member DataFrame (after loading or generating):")
print(member_df.head())

# Process Member Data
updated_members = []

for _, member in member_df.iterrows():
    policy_id = member['Policy ID']
    policy_data = policy_df[policy_df['Policy ID'] == policy_id]

    if policy_data.empty:
        print(f"No matching policy found for Policy ID: {policy_id}")
        continue

    policy_data = policy_data.iloc[0]
    location = policy_data['Location Type']
    policy_type = policy_data['Policy Type']
    num_members = policy_data['Number of Members']
    member_index = int(member["Member ID"].split("-")[-1])

    # Assign attributes
    marital_status = assign_marital_status(location)
    designation = assign_designation(policy_type)
    relationship = assign_relationship(num_members, member_index)
    is_child = relationship == "Child"
    gender = assign_gender(location, is_child)
    age = assign_age(policy_type, relationship)
    income = assign_income(policy_type, designation, location, age)
    chronic_conditions = assign_chronic_conditions()
    smoking_status = assign_smoking_status()
    alcohol_consumption = assign_alcohol_consumption()

    # Create updated member record
    updated_member = {
        **member.to_dict(),
        "Member Marital Status": marital_status,
        "Member Designation": designation,
        "Member Relationship": relationship,
        "Member Gender": gender,
        "Member Age": age,
        "Member Income": income,
        "Member Chronic Conditions": chronic_conditions,
        "Member Smoking Status": smoking_status,
        "Member Alcohol Consumption": alcohol_consumption
    }
    updated_members.append(updated_member)

# Convert updated members back to DataFrame
if updated_members:
    member_df = pd.DataFrame(updated_members)
    print("Updated Member DataFrame:")
    print(member_df.head())
else:
    print("No members were updated. Check data consistency.")

Member DataFrame (after loading or generating):
  Policy ID  Policyholder ID Member ID Member Ethnicity Member Status
0  10000000         957299.0       M-1          Unknown       Unknown
1  10000001         957300.0       M-2          Unknown       Unknown
2  10000002         957301.0       M-3          Unknown       Unknown
3  10000003         957302.0       M-4          Unknown       Unknown
4  10000004         957303.0       M-5          Unknown       Unknown


In [8]:
member_df.shape

(5, 14)

#### Populate Column Values for columns Member Marital Status, Member Designation, Member Relationship, Member Gender, Member Age, Member Income, Member Chronic Conditions, Member Smoking Status, and Member Alcohol Consumption

In [9]:
import pandas as pd
import numpy as np

# Helper Functions (same as before)
# Define all assign_... functions here

# Sample DataFrames
policy_df = pd.DataFrame({
    "Policy ID": ["10000000", "10000001", "10000002", "10000003", "10000004"],
    "Policyholder ID": [957299, 957300, 957301, 957302, 957303],
    "Location Type": ["Urban", "Rural", "Suburban", "Suburban", "Urban"],
    "Policy Type": ["Group Health Plan"] * 5,
    "Number of Members": [1, 1, 1, 1, 1]
})

policy_df['Policy ID'] = policy_df['Policy ID'].astype(str)

member_df = policy_df[['Policy ID', 'Policyholder ID']].copy()
member_df['Member ID'] = ['M-' + str(i) for i in range(1, len(member_df) + 1)]
member_df['Member Ethnicity'] = ['Unknown'] * len(member_df)
member_df['Member Status'] = ['Unknown'] * len(member_df)

# Debugging: Initial Member DataFrame
print("Initial Member DataFrame:")
print(member_df)

# Process Member Data
updated_members = []

for _, member in member_df.iterrows():
    policy_id = member['Policy ID']
    policy_data = policy_df[policy_df['Policy ID'] == policy_id]

    if policy_data.empty:
        print(f"No matching policy found for Policy ID: {policy_id}")
        continue

    policy_data = policy_data.iloc[0]
    location = policy_data['Location Type']
    policy_type = policy_data['Policy Type']
    num_members = policy_data['Number of Members']
    member_index = int(member["Member ID"].split("-")[-1])

    # Assign attributes
    marital_status = assign_marital_status(location)
    designation = assign_designation(policy_type)
    relationship = assign_relationship(num_members, member_index)
    is_child = relationship == "Child"
    gender = assign_gender(location, is_child)
    age = assign_age(policy_type, relationship)
    income = assign_income(policy_type, designation, location, age)
    chronic_conditions = assign_chronic_conditions()
    smoking_status = assign_smoking_status()
    alcohol_consumption = assign_alcohol_consumption()

    # Create updated member record
    updated_member = {
        **member.to_dict(),
        "Member Marital Status": marital_status,
        "Member Designation": designation,
        "Member Relationship": relationship,
        "Member Gender": gender,
        "Member Age": age,
        "Member Income": income,
        "Member Chronic Conditions": chronic_conditions,
        "Member Smoking Status": smoking_status,
        "Member Alcohol Consumption": alcohol_consumption
    }

    # Debug: Print the updated record
    print("Updated Member Record:")
    print(updated_member)

    updated_members.append(updated_member)

# Convert updated members back to DataFrame
if updated_members:
    member_df = pd.DataFrame(updated_members)
    print("Final Member DataFrame Columns:")
    print(member_df.columns)
    print("Updated Member DataFrame:")
    print(member_df)
else:
    print("No members were updated. Check data consistency.")


Initial Member DataFrame:
  Policy ID  Policyholder ID Member ID Member Ethnicity Member Status
0  10000000           957299       M-1          Unknown       Unknown
1  10000001           957300       M-2          Unknown       Unknown
2  10000002           957301       M-3          Unknown       Unknown
3  10000003           957302       M-4          Unknown       Unknown
4  10000004           957303       M-5          Unknown       Unknown
Updated Member Record:
{'Policy ID': '10000000', 'Policyholder ID': 957299, 'Member ID': 'M-1', 'Member Ethnicity': 'Unknown', 'Member Status': 'Unknown', 'Member Marital Status': 'Married', 'Member Designation': 'Support Staff', 'Member Relationship': 'Policyholder', 'Member Gender': 'Female', 'Member Age': 43, 'Member Income': 22186, 'Member Chronic Conditions': 'None', 'Member Smoking Status': 'Non-Smoker', 'Member Alcohol Consumption': 'None'}
Updated Member Record:
{'Policy ID': '10000001', 'Policyholder ID': 957300, 'Member ID': 'M-2', 'Membe

In [10]:
member_df.shape

(5, 14)

In [ ]:
member_df.columns

Index(['Policy ID', 'Policyholder ID', 'Policy Start Date', 'Policy End Date',
       'Churn Flag', 'Renewal Flag', 'Member ID', 'Member Ethnicity',
       'Member Status', 'Member Inclusion Date', 'Member Removal Date'],
      dtype='object')

#### Populate Column Values for columns Member Physical Activity Level, Member Height, Member Weight, Member BMI Category, Member Disabilities, Member Claim History, Member Claim Frequency, Member Risk Score, Member Tenure, Member Productivity Index, Member Work Hours



In [ ]:
# Helper Functions for New Columns

def assign_physical_activity_level():
    """Assign physical activity level."""
    return np.random.choice(["Sedentary", "Light", "Moderate", "High"], p=[0.3, 0.4, 0.2, 0.1])

def assign_height(gender, age):
    """Assign height based on gender and age."""
    if age <= 18:  # Child
        return np.random.randint(100, 170)  # Height in cm
    if gender == "Male":
        return np.random.randint(160, 200)
    elif gender == "Female":
        return np.random.randint(150, 180)
    return np.random.randint(150, 190)  # Default for "Other"

def assign_weight(height, age, chronic_conditions):
    """Assign weight based on height, age, and chronic conditions."""
    bmi = np.random.normal(22, 3)  # Normal BMI with some variability
    if "Hypertension" in chronic_conditions or "Diabetes" in chronic_conditions:
        bmi += np.random.uniform(2, 5)
    if age > 50:
        bmi += np.random.uniform(1, 3)
    weight = (bmi * (height / 100) ** 2)  # Calculate weight from BMI and height
    return int(weight)

def calculate_bmi_category(weight, height):
    """Calculate BMI category."""
    bmi = weight / (height / 100) ** 2
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 24.9:
        return "Normal"
    elif 25 <= bmi < 29.9:
        return "Overweight"
    else:
        return "Obese"

def assign_disabilities(age, chronic_conditions, industry=None):
    """Assign disabilities based on age, chronic conditions, and industry."""
    base_probabilities = {"None": 90, "Physical": 5, "Cognitive": 3, "Both": 2}
    if age > 50:
        base_probabilities["Physical"] += 3
        base_probabilities["Both"] += 2
    if industry == "Manufacturing":
        base_probabilities["Physical"] += 2
    elif industry in ["Finance", "Tech"]:
        base_probabilities["Cognitive"] += 1
    if "Hypertension" in chronic_conditions or "Diabetes" in chronic_conditions:
        base_probabilities["Both"] += 3
    choices, weights = zip(*base_probabilities.items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_claim_history(age, chronic_conditions):
    """Assign claim history."""
    base_probabilities = {"No Claims": 80, "Minor Claims": 15, "Major Claims": 5}
    if age > 50:
        base_probabilities["Minor Claims"] += 5
        base_probabilities["Major Claims"] += 2
    if "Diabetes" in chronic_conditions or "Hypertension" in chronic_conditions:
        base_probabilities["Minor Claims"] += 7
    if "Combination" in chronic_conditions:
        base_probabilities["Major Claims"] += 10
    choices, weights = zip(*base_probabilities.items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_claim_frequency(claim_history, chronic_conditions):
    """Assign claim frequency based on history and conditions."""
    if claim_history == "No Claims":
        return 0
    elif claim_history == "Minor Claims":
        return np.random.choice([1, 2], p=[0.8, 0.2])
    elif claim_history == "Major Claims":
        return np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])

def calculate_risk_score(age, chronic_conditions, claim_history, smoking_status, physical_activity_level, alcohol_consumption):
    """Calculate risk score."""
    score = 0
    score += (age - 30) // 5
    score += 5 * chronic_conditions.count("None") if chronic_conditions != "None" else 0
    score += 3 if claim_history == "Minor Claims" else 5 if claim_history == "Major Claims" else 0
    score += 5 if smoking_status == "Smoker" else 0
    score += 3 if physical_activity_level == "Sedentary" else 0
    score += 2 if alcohol_consumption == "High" else 0
    return score

# Add New Columns to Member Dataset
updated_members = []
for _, member in member_df.iterrows():
    # Fetch existing data
    gender = member["Member Gender"]
    age = member["Member Age"]
    chronic_conditions = member["Member Chronic Conditions"]
    industry = member.get("Industry Type", None)

    # Calculate additional fields
    physical_activity_level = assign_physical_activity_level()
    height = assign_height(gender, age)
    weight = assign_weight(height, age, chronic_conditions)
    bmi_category = calculate_bmi_category(weight, height)
    disabilities = assign_disabilities(age, chronic_conditions, industry)
    claim_history = assign_claim_history(age, chronic_conditions)
    claim_frequency = assign_claim_frequency(claim_history, chronic_conditions)
    risk_score = calculate_risk_score(age, chronic_conditions, claim_history, member["Member Smoking Status"], physical_activity_level, member["Member Alcohol Consumption"])

    # Update member data
    member.update({
        "Member Physical Activity Level": physical_activity_level,
        "Member Height": height,
        "Member Weight": weight,
        "Member BMI Category": bmi_category,
        "Member Disabilities": disabilities,
        "Member Claim History": claim_history,
        "Member Claim Frequency": claim_frequency,
        "Member Risk Score": risk_score
    })
    updated_members.append(member)

# Update the DataFrame
member_df = pd.DataFrame(updated_members)

member_df.head()

KeyError: 'Member Gender'

In [ ]:
member_df.shape

#### Writing the Synthetic Policy Dataset to a CSV File and then Downloaded


In [ ]:
# Save the dataset to a local file in Colab
file_path = 'inpatient_healthcare_insurance_synthetic_member_dataset.csv'
member_df.to_csv(file_path, index=False)
print(f"Dataset saved temporarily at {file_path}")

Dataset saved temporarily at inpatient_healthcare_insurance_synthetic_member_dataset.csv


In [ ]:
from google.colab import files

# Download the file
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>